In [8]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [10]:
# ==============================================================================
# 1. INPUTS DEL USUARIO
# ==============================================================================
sigma1 = 0.20       # Riesgo/Volatilidad Activo 1 (20%)
sigma2 = 0.35       # Riesgo/Volatilidad Activo 2 (35%)
rho12  = 0.30       # Correlación entre Activo 1 y Activo 2

E1 = 0.08           # Retorno esperado Activo 1 (8%)
E2 = 0.15           # Retorno esperado Activo 2 (15%)
mu_objetivo = 0.11  # Retorno esperado objetivo (11%)
N_muestras = 1000   # Número de datos simulados (Ajustable)

In [11]:
# ==============================================================================
# 2. CONSTRUCCIÓN DE MATRICES Y HESSIANA
# ==============================================================================
cov12 = rho12 * sigma1 * sigma2
Sigma = np.array([
    [sigma1**2, cov12],
    [cov12, sigma2**2]
])

# La matriz Hessiana de L respecto a (w1, w2) es H = 2 * Sigma
H = 2 * Sigma

# Autovalores y Autovectores de Σ
val_S, vec_S = np.linalg.eigh(Sigma)
lambda_s1, lambda_s2 = val_S[0], val_S[1]

# Autovalores y Autovectores de H
val_H, vec_H = np.linalg.eigh(H)
lambda_h1, lambda_h2 = val_H[0], val_H[1]

u1 = vec_H[:, 0]  # Primer eje principal
u2 = vec_H[:, 1]  # Segundo eje principal

# Radios de curvatura principales
R1 = 1.0 / lambda_h1
R2 = 1.0 / lambda_h2
det_H = np.linalg.det(H)

In [12]:
# ==============================================================================
# 3. RESOLUCIÓN DE LA LAGRANGIANA (Punto Crítico Restringido)
# ==============================================================================
# Sistema lineal A * [w1, w2, lambda1, lambda2]^T = b
A = np.array([
    [2*Sigma[0,0], 2*Sigma[0,1], -E1, -1.0],
    [2*Sigma[1,0], 2*Sigma[1,1], -E2, -1.0],
    [E1,           E2,            0.0,  0.0],
    [1.0,          1.0,           0.0,  0.0]
])

b = np.array([0.0, 0.0, mu_objetivo, 1.0])
sol = np.linalg.solve(A, b)

w1_opt, w2_opt = sol[0], sol[1]
l1_opt, l2_opt = sol[2], sol[3]

# Función evaluadora de la Lagrangiana
def L_func(w1, w2):
    return (w1**2 * Sigma[0,0] + 2*w1*w2*Sigma[0,1] + w2**2 * Sigma[1,1] 
            - l1_opt*(w1*E1 + w2*E2 - mu_objetivo) - l2_opt*(w1 + w2 - 1.0))

L_opt = L_func(w1_opt, w2_opt)

In [13]:
# ==============================================================================
# 3. CURVAS SOBRE LA SUPERFICIE (PLANOS VERTICALES EN LOS EJES PRINCIPALES)
# ==============================================================================
t_vals = np.linspace(-0.8, 0.8, 100)

# Curva 1: Se extiende a lo largo del eje u1 centrado en (w1_opt, w2_opt)
w1_c1 = w1_opt + t_vals * u1[0]
w2_c1 = w2_opt + t_vals * u1[1]
L_c1   = L_func(w1_c1, w2_c1)

# Curva 2: Se extiende a lo largo del eje u2 centrado en (w1_opt, w2_opt)
w1_c2 = w1_opt + t_vals * u2[0]
w2_c2 = w2_opt + t_vals * u2[1]
L_c2   = L_func(w1_c2, w2_c2)

In [14]:
print("--- RESULTADOS Y ANÁLISIS ---")
print(f"Matriz Hessiana H:\n{H}")
print(f"Autovalores de H: λ1 = {lambda_h1:.4f}, λ2 = {lambda_h2:.4f}")
print(f"Determinante det(H): {det_H:.6f}")
print(f"Radios de Curvatura Principales: R1 = {R1:.4f}, R2 = {R2:.4f}")
print(f"\nPesos Óptimos Restringidos: w1* = {w1_opt:.4f}, w2* = {w2_opt:.4f}")
print(f"Multiplicadores de Lagrange: λ1* = {l1_opt:.4f}, λ2* = {l2_opt:.4f}")

--- RESULTADOS Y ANÁLISIS ---
Matriz Hessiana H:
[[0.08  0.042]
 [0.042 0.245]]
Autovalores de H: λ1 = 0.0699, λ2 = 0.2551
Determinante det(H): 0.017836
Radios de Curvatura Principales: R1 = 14.3012, R2 = 3.9204

Pesos Óptimos Restringidos: w1* = 0.5714, w2* = 0.4286
Multiplicadores de Lagrange: λ1* = 0.9327, λ2* = -0.0109


In [ ]:
# ==============================================================================
# 4. VISUALIZACIÓN 3D CON PLOTLY
# ==============================================================================
w1_grid = np.linspace(w1_opt - 1.0, w1_opt + 1.0, 80)
w2_grid = np.linspace(w2_opt - 1.0, w2_opt + 1.0, 80)
W1, W2 = np.meshgrid(w1_grid, w2_grid)
L_surface = L_func(W1, W2)

fig = go.Figure()

# Superficie Principal L(w1, w2)
fig.add_trace(go.Surface(
    x=W1, y=W2, z=L_surface,
    colorscale='Viridis', opacity=0.75,
    name='L(w₁, w₂)', showscale=False
))

# Curva sobre la superficie a lo largo del Eje Principal u1
fig.add_trace(go.Scatter3d(
    x=w1_c1, y=w2_c1, z=L_c1,
    mode='lines', line=dict(color='red', width=7),
    name=f'Curva Eje u₁ (λ₁ = {lambda_h1:.3f})'
))

# Curva sobre la superficie a lo largo del Eje Principal u2
fig.add_trace(go.Scatter3d(
    x=w1_c2, y=w2_c2, z=L_c2,
    mode='lines', line=dict(color='cyan', width=7),
    name=f'Curva Eje u₂ (λ₂ = {lambda_h2:.3f})'
))

# Vectores Principales Centrados en el Punto Mínimo (Proyección en el piso)
scale_vec = 0.6
fig.add_trace(go.Scatter3d(
    x=[w1_opt - scale_vec*u1[0], w1_opt + scale_vec*u1[0]],
    y=[w2_opt - scale_vec*u1[1], w2_opt + scale_vec*u1[1]],
    z=[L_opt, L_opt],
    mode='lines+text', line=dict(color='darkred', width=4, dash='dash'),
    text=["", "Eje u₁"], name='Dirección Principal u₁'
))

fig.add_trace(go.Scatter3d(
    x=[w1_opt - scale_vec*u2[0], w1_opt + scale_vec*u2[0]],
    y=[w2_opt - scale_vec*u2[1], w2_opt + scale_vec*u2[1]],
    z=[L_opt, L_opt],
    mode='lines+text', line=dict(color='darkblue', width=4, dash='dash'),
    text=["", "Eje u₂"], name='Dirección Principal u₂'
))

# Marcador del Mínimo Óptimo Restringido
fig.add_trace(go.Scatter3d(
    x=[w1_opt], y=[w2_opt], z=[L_opt],
    mode='markers+text', marker=dict(size=8, color='magenta', symbol='diamond'),
    text=[f" Mínimo ({w1_opt:.2f}, {w2_opt:.2f})"],
    textposition="top center", name='Mínimo Óptimo'
))

# Layout y Cámara
fig.update_layout(
    title=dict(
        text=f"Curvaturas Principales centradas en el Mínimo de la Lagrangiana<br><sup>λ₁ = {lambda_h1:.3f} | λ₂ = {lambda_h2:.3f} | R₁ = {1/lambda_h1:.2f}, R₂ = {1/lambda_h2:.2f}</sup>",
        x=0.5
    ),
    scene=dict(
        xaxis_title='w₁',
        yaxis_title='w₂',
        zaxis_title='Lagrangiana L',
        camera=dict(eye=dict(x=1.4, y=1.4, z=1.1))
    ),
    margin=dict(l=20, r=20, b=20, t=70)
)

fig.show()

In [15]:
# Malla para las curvas de nivel del gráfico 1
w1_grid = np.linspace(w1_opt - 1.5, w1_opt + 1.5, 100)
w2_grid = np.linspace(w2_opt - 1.5, w2_opt + 1.5, 100)
W1, W2 = np.meshgrid(w1_grid, w2_grid)
L_surface = L_func(W1, W2)

# ==============================================================================
# 4. SIMULACIÓN DE DATOS CON COVARIANZA Sigma
# ==============================================================================
np.random.seed(42)
mean_sim = np.array([0.0, 0.0])  # Centrado en el origen
datos_simulados = np.random.multivariate_normal(mean_sim, Sigma, N_muestras)

# ==============================================================================
# 5. CREACIÓN DE SUBPLOTS LADO A LADO
# ==============================================================================
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"<b>1. Hessiana H = 2Σ (Curvatura Lagrangiana)</b><br><sup>λ_H1 = {lambda_h1:.4f} | λ_H2 = {lambda_h2:.4f}</sup>",
        f"<b>2. N={N_muestras} Puntos Simulados con Covarianza Σ</b><br><sup>λ_Σ1 = {lambda_s1:.4f} | λ_Σ2 = {lambda_s2:.4f} (Factor 1/2)</sup>"
    ),
    horizontal_spacing=0.10
)

# ------------------------------------------------------------------------------
# PANEL IZQUIERDO: HESSIANA Y CURVAS DE NIVEL
# ------------------------------------------------------------------------------
# Curvas de Nivel
fig.add_trace(go.Contour(
    x=w1_grid, y=w2_grid, z=L_surface,
    colorscale='Viridis',
    contours=dict(coloring='heatmap', showlabels=True, labelfont=dict(size=10, color='white')),
    showscale=False,
    name='Curvas L(w)'
), row=1, col=1)

# Ejes principales H
s_h1 = 0.4 / np.sqrt(lambda_h1)
s_h2 = 0.4 / np.sqrt(lambda_h2)

fig.add_trace(go.Scatter(
    x=[w1_opt, w1_opt + s_h1 * u1[0]], y=[w2_opt, w2_opt + s_h1 * u1[1]],
    mode='lines+markers+text', line=dict(color='red', width=3), marker=dict(size=6),
    text=["", f"u₁ (λ_H1={lambda_h1:.3f})"], textposition="top left",
    textfont=dict(color='white', size=10, family="Arial Black"), name='Eje u₁ (Hessiana)'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=[w1_opt, w1_opt + s_h2 * u2[0]], y=[w2_opt, w2_opt + s_h2 * u2[1]],
    mode='lines+markers+text', line=dict(color='cyan', width=3), marker=dict(size=6),
    text=["", f"u₂ (λ_H2={lambda_h2:.3f})"], textposition="top right",
    textfont=dict(color='white', size=10, family="Arial Black"), name='Eje u₂ (Hessiana)'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=[w1_opt], y=[w2_opt], mode='markers+text',
    marker=dict(size=10, color='magenta', symbol='diamond'),
    text=["Mínimo  "], textposition="bottom left",
    textfont=dict(color='white', size=10, family="Arial Black"), name='Punto Óptimo'
), row=1, col=1)

# ------------------------------------------------------------------------------
# PANEL DERECHO: PUNTOS SIMULADOS Y MATRIZ DE COVARIANZA
# ------------------------------------------------------------------------------
# N Puntos Simulados
fig.add_trace(go.Scatter(
    x=datos_simulados[:, 0], y=datos_simulados[:, 1],
    mode='markers', marker=dict(size=4, color='rgba(0, 120, 255, 0.45)'),
    name='Puntos Simulados'
), row=1, col=2)

# Ejes principales Σ (Escalados por 2 desviaciones estándar 2*sqrt(λ) para abarcar los datos)
s_s1 = 2.0 * np.sqrt(lambda_s1)
s_s2 = 2.0 * np.sqrt(lambda_s2)

fig.add_trace(go.Scatter(
    x=[0, s_s1 * u1[0]], y=[0, s_s1 * u1[1]],
    mode='lines+markers+text', line=dict(color='red', width=3), marker=dict(size=6),
    text=["", f"u₁ (λ_Σ1={lambda_s1:.3f})"], textposition="top left",
    textfont=dict(color='black', size=10, family="Arial Black"), name='Eje u₁ (Covarianza)'
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=[0, s_s2 * u2[0]], y=[0, s_s2 * u2[1]],
    mode='lines+markers+text', line=dict(color='cyan', width=3), marker=dict(size=6),
    text=["", f"u₂ (λ_Σ2={lambda_s2:.3f})"], textposition="top right",
    textfont=dict(color='black', size=10, family="Arial Black"), name='Eje u₂ (Covarianza)'
), row=1, col=2)

# ------------------------------------------------------------------------------
# CONFIGURACIÓN GENERAL DEL LAYOUT
# ------------------------------------------------------------------------------
fig.update_layout(
    width=1100, height=580,
    title=dict(
        text="<b>Comparativa: Curvatura Hessiana H = 2Σ vs. Dispersión Muestral Σ</b><br><sup>Relación Directa: λ_H = 2 · λ_Σ | Direcciones u₁, u₂ son Rigurosamente Idénticas</sup>",
        x=0.5, xanchor='center'
    ),
    legend=dict(
        orientation="h", yanchor="top", y=-0.12, xanchor="center", x=0.5,
        bgcolor="rgba(255, 255, 255, 0.9)", bordercolor="lightgray", borderwidth=1
    ),
    margin=dict(l=50, r=50, b=100, t=100)
)

fig.update_xaxes(title_text="w₁", row=1, col=1, scaleanchor="y1", scaleratio=1)
fig.update_yaxes(title_text="w₂", row=1, col=1)

fig.update_xaxes(title_text="X₁ (Rendimiento Activo 1)", row=1, col=2, scaleanchor="y2", scaleratio=1)
fig.update_yaxes(title_text="X₂ (Rendimiento Activo 2)", row=1, col=2)

fig.show()

# Print explicativo en consola
print("=== VERIFICACIÓN MATEMÁTICA DE ESCALA ===")
print(f"Autovalores de Σ:       λ_Σ1 = {lambda_s1:.6f},  λ_Σ2 = {lambda_s2:.6f}")
print(f"Autovalores de H (2Σ):  λ_H1 = {lambda_h1:.6f},  λ_H2 = {lambda_h2:.6f}")
print(f"Razón (λ_H / λ_Σ):      [{lambda_h1/lambda_s1:.1f}x, {lambda_h2/lambda_s2:.1f}x]")

=== VERIFICACIÓN MATEMÁTICA DE ESCALA ===
Autovalores de Σ:       λ_Σ1 = 0.034962,  λ_Σ2 = 0.127538
Autovalores de H (2Σ):  λ_H1 = 0.069924,  λ_H2 = 0.255076
Razón (λ_H / λ_Σ):      [2.0x, 2.0x]


In [71]:
import numpy as np
import plotly.graph_objects as go

# ==============================================================================
# PARAMETROS CONFIGURABLES
# ==============================================================================
# Ángulo de rotación theta = 30 grados
theta = np.pi / 6  

# Autovalor lambda_2 fijo
lambda2_fixed = 0.0  # <--- Modifica este valor para cambiar lambda_2

# Rango de autovalores para lambda_1
L_max = 5.0
lambda_range = np.linspace(-L_max, L_max, 21)
# ==============================================================================

# 1. Definir los ejes principales fijos
u1 = np.array([np.cos(theta), np.sin(theta)])   # Primer autovector principal
u2 = np.array([-np.sin(theta), np.cos(theta)])  # Segundo autovector principal
Q = np.column_stack((u1, u2))

# Malla de pesos (w1, w2) para la superficie
w1_vals = np.linspace(-2, 2, 80)
w2_vals = np.linspace(-2, 2, 80)
W1, W2 = np.meshgrid(w1_vals, w2_vals)

# Transformar coordenadas a la base de ejes principales
Y1 = W1 * Q[0, 0] + W2 * Q[1, 0]
Y2 = W1 * Q[0, 1] + W2 * Q[1, 1]

def compute_surface_and_matrix(l1, l2):
    # Función de forma cuadrática: f(w) = lambda1 * y1^2 + lambda2 * y2^2
    Z = l1 * (Y1**2) + l2 * (Y2**2)
    
    # Matriz Sigma = Q * diag(l1, l2) * Q^T
    Lambda = np.diag([l1, l2])
    Sigma = Q @ Lambda @ Q.T
    det_Sigma = l1 * l2
    
    # Expresión algebraica
    s11, s12, s22 = Sigma[0, 0], Sigma[0, 1], Sigma[1, 1]
    formula_str = f"f(w₁, w₂) = {s11:.2f}w₁² + {2*s12:.2f}w₁w₂ + {s22:.2f}w₂²"
    
    return Z, Sigma, det_Sigma, formula_str

# Función auxiliar para determinar la clasificación geométrica
def get_surface_type(l1, l2):
    det = l1 * l2
    if det > 0:
        return "Paraboloide (Mínimo)" if l1 > 0 else "Paraboloide (Máximo)"
    elif det < 0:
        return "Silla de Montar (Saddle)"
    else:
        return "Cilindro Parabólico" if (l1 != 0 or l2 != 0) else "Plano (Origen)"

# 2. Inicializar la figura con Plotly
fig = go.Figure()

# Plot inicial con lambda1 = 1.0 y el lambda2_fixed parametrizado
lambda1_init = 1.0
Z0, Sigma0, det_S0, formula0 = compute_surface_and_matrix(lambda1_init, lambda2_fixed)
tipo0 = get_surface_type(lambda1_init, lambda2_fixed)

# Traza 0: Superficie
fig.add_trace(go.Surface(
    x=W1, y=W2, z=Z0,
    colorscale='Viridis',
    opacity=0.85,
    name='Superficie Cuadrática',
    showscale=False
))

# Traza 1: Autovector 1 (Eje principal 1)
scale_vector = 1.5
fig.add_trace(go.Scatter3d(
    x=[0, scale_vector * u1[0]],
    y=[0, scale_vector * u1[1]],
    z=[0, 0],
    mode='lines+markers+text',
    line=dict(color='red', width=6),
    marker=dict(size=4),
    text=["", "Eje Principal u₁"],
    name='Eje Principal 1 (u₁)'
))

# Traza 2: Autovector 2 (Eje principal 2)
fig.add_trace(go.Scatter3d(
    x=[0, scale_vector * u2[0]],
    y=[0, scale_vector * u2[1]],
    z=[0, 0],
    mode='lines+markers+text',
    line=dict(color='blue', width=6),
    marker=dict(size=4),
    text=["", "Eje Principal u₂"],
    name='Eje Principal 2 (u₂)'
))

# Traza 3: Punto crítico en el origen
fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode='markers',
    marker=dict(size=7, color='gold', symbol='diamond'),
    name='Origen (0,0)'
))

# 3. Construir pasos para el Slider evaluando con lambda2_fixed
steps_l1 = []
for l1 in lambda_range:
    Z, Sigma, det_S, formula = compute_surface_and_matrix(l1, lambda2_fixed)
    tipo = get_surface_type(l1, lambda2_fixed)

    title_text = (
        f"Formas Cuadráticas (θ = 30°, λ₂ = {lambda2_fixed:.1f})<br>"
        f"<sup>{formula} | det(Σ) = {det_S:.2f} | Tipo: {tipo}</sup>"
    )

    step = dict(
        method="update",
        args=[
            {"z": [Z]},                  # Restyle: actualiza la matriz Z
            {"title.text": title_text},  # Relayout: actualiza el título dinámicamente
            [0]                          # Aplica solo a la traza 0 (Superficie)
        ],
        label=f"{l1:.1f}"
    )
    steps_l1.append(step)

# 4. Configuración del Layout
initial_title = (
    f"Formas Cuadráticas (θ = 30°, λ₂ = {lambda2_fixed:.1f})<br>"
    f"<sup>{formula0} | det(Σ) = {det_S0:.2f} | Tipo: {tipo0}</sup>"
)

fig.update_layout(
    title=dict(
        text=initial_title,
        x=0.5
    ),
    scene=dict(
        xaxis=dict(title='Peso w₁', range=[-2, 2]),
        yaxis=dict(title='Peso w₂', range=[-2, 2]),
        zaxis=dict(title='f(w₁, w₂)', range=[-25, 25]),
        aspectmode='cube', 
        camera=dict(eye=dict(x=1.6, y=1.6, z=1.2))
    ),
    sliders=[
        dict(
            active=12,  # Posición inicial (~1.0 en el rango)
            currentvalue={"prefix": "Autovalor λ₁: "},
            pad={"t": 50},
            steps=steps_l1
        )
    ],
    margin=dict(l=20, r=20, b=20, t=80)
)

fig.show()